In [1]:
import os, requests, json, pyodbc, logging
import pandas as pd
from datetime import datetime, timezone
from dotenv import load_dotenv
from sqlalchemy import create_engine
from tqdm import tqdm
from datetime import datetime, timedelta, time


load_dotenv()

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

API_KEY = os.getenv("INAGENT_API_KEY")
ENDPOINT = os.getenv("INAGENT_URL")
CREW_ID = os.getenv("INAGENT_CREW_ID")

In [2]:
hoy = datetime.now()
ayer = hoy - timedelta(days=1)

#start_date = ayer.strftime("%Y-%m-%dT00:00:00")
end_date = ayer.strftime("%Y-%m-%dT23:59:59")

In [3]:
def to_unix_ms(iso_date):
    dt = datetime.fromisoformat(iso_date).replace(tzinfo=timezone.utc)
    return int(dt.timestamp() * 1000)

def safe_json_parse(val):
    try:
        return json.loads(val) if (val and val != 'null') else {}
    except:
        return {}

print(" Herramientas listas: to_unix_ms y safe_json_parse.")

 Herramientas listas: to_unix_ms y safe_json_parse.


In [4]:
start_date = "2026-03-18T00:00:00"
#end_date = "2026-03-25T23:59:59"

all_data = []
page = 0
page_size = 100 

while True:
    params = {
        "crew_id": CREW_ID,
        "start_ts": to_unix_ms(start_date),
        "end_ts": to_unix_ms(end_date),
        "page": page,
        "pageSize": page_size
    }
    
    headers = {"apikey": API_KEY} # Header de autenticación obligatorio [cite: 19, 36]
    res = requests.get(ENDPOINT, headers=headers, params=params)
    
    if res.status_code != 200:
        logger.error(f"Fallo en página {page}: {res.text}")
        break
        
    data_payload = res.json().get("data", {})
    rows = data_payload.get("rows", []) # Registros posicionales [cite: 86, 124]
    cols = data_payload.get("dataSchema", {}).get("columnNames", []) # Nombres de columnas [cite: 45, 123]
    
    if not rows:
        break
        
    all_data.extend(rows)
    print(f"Página {page} procesada. Total acumulado: {len(all_data)}")
    
    # Si recibimos menos de 100, significa que llegamos al final (ej. el registro 105) [cite: 144]
    if len(rows) < page_size:
        break
    page += 1

df_raw = pd.DataFrame(all_data, columns=cols)
display(df_raw.head())

Página 0 procesada. Total acumulado: 46


,Id,Inicio,Fin,Duración (s),Análisis Sentimental,Tema general de la conversación,Resumen,Es saliente,Motivo de Cierre,Mensajes del asistente,...,Motivo de abandono,Fue abandonada,Fue transferida,Transferida a,Motivo de transferencia,Fue solo agradecimiento,Id Externo,Id Canal,Canal,Timestamp
0,aa106f71-c43c-4483-8340-7d868d8d770a,2026-03-23T17:23:06.000Z,2026-03-23T17:25:44.959Z,158,neutral,cita dental,"Usuario quiere cancelar cita dental, proporcio...",False,finished,6,...,None,False,None,None,,False,,NaN,,1774286586000
1,6a2c0317-58b3-4326-bf13-2e98dcdabfa9,2026-03-23T17:21:01.000Z,2026-03-23T17:21:42.770Z,41,neutral,Transferencia Asesor,Usuario solicita transferencia a asesor sin pr...,False,finished,2,...,None,False,None,None,El cliente solicitó ser transferido directamen...,False,,NaN,,1774286461000
2,fadce121-be05-44ff-940c-bc58f02d1b5c,2026-03-23T17:04:51.000Z,2026-03-23T17:10:37.814Z,346,neutral,citas dentales,Usuario proporcionó tarjeta y datos para agend...,False,finished,13,...,None,False,None,None,,False,,NaN,,1774285491000
3,9e54e341-7ea7-4234-8262-d04904e84e60,2026-03-23T17:00:44.000Z,2026-03-23T17:04:43.083Z,239,neutral,Tarjeta Liverpool,Usuario proporciona números de tarjeta Liverpo...,False,finished,4,...,None,False,None,None,,False,,NaN,,1774285244000
4,e06aa9a6-9ef6-4d32-a344-0837a2387b33,2026-03-23T16:15:22.000Z,2026-03-23T16:22:39.473Z,437,neutral,Cita dental,Usuario agenda y luego cancela cita dental con...,False,finished,16,...,None,False,None,None,,False,,NaN,,1774282522000


In [5]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 46 entries, 0 to 45
Data columns (total 30 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Id                               46 non-null     str    
 1   Inicio                           46 non-null     str    
 2   Fin                              46 non-null     str    
 3   Duración (s)                     46 non-null     int64  
 4   Análisis Sentimental             16 non-null     str    
 5   Tema general de la conversación  16 non-null     str    
 6   Resumen                          16 non-null     str    
 7   Es saliente                      46 non-null     bool   
 8   Motivo de Cierre                 45 non-null     str    
 9   Mensajes del asistente           46 non-null     int64  
 10  Mensajes del usuario             46 non-null     int64  
 11  Prom. Tpo Respuesta Asistente    46 non-null     int64  
 12  Prom. Tpo Respuesta Usuario      46

In [6]:
native_whitelist = [
    'Id', 
    'Id Externo',
    'Id Canal',
    'Canal',
    'Timestamp',
    'Inicio',
    'Fin',
    'Duración (s)', 
    'Análisis Sentimental',
    'Tema general de la conversación',
    'Resumen',
    'Fue resuelta',
    'Fue solo agradecimiento',
    
    'Mensajes del asistente',
    'Mensajes del usuario',
    'Cantidad de preguntas a QNA',
    'Herramientas Usadas',
    
    'Es saliente',
    'Motivo de Cierre',
    'Cerrada por inactividad',
    'Fue abandonada',
    'Motivo de transferencia'
    
    # 'Moderadores_Usados',
    # 'Modificadores_Usados',
    # 'Motivo_de_abandono',
    # 'Fue_transferida',
    # 'Transferida_a'
]

In [7]:
context_whitelist = [
    'creationRequestTimestamp',
    'startedTime',
    'startDelayMilliseconds',
    'API_KEY',
    'URL',
    'tarjeta',
    'siniestralidad',
    'comb_resume', # Cuidado: Solo 5 registros
    'proxyData_recordFile',
    'proxyData_roomId',
    'proxyData_roomName',
    'toolLogs'
]

In [8]:
df_native = df_raw[[c for c in native_whitelist if c in df_raw.columns]].copy()

ctx_raw = pd.json_normalize(df_raw['Contexto'].apply(safe_json_parse))
ctx_raw.columns = [c.replace(".", "_") for c in ctx_raw.columns]

ctx_selected = ctx_raw[[c for c in context_whitelist if c in ctx_raw.columns]].add_prefix('ctx_')
df_base = pd.concat([df_native, ctx_selected], axis=1)
print(f"{df_base.shape}")

(46, 34)


In [9]:
col_logs = 'ctx_toolLogs'

df_tools_exploded = df_base[['Id', col_logs]].dropna(subset=[col_logs]).explode(col_logs)
tool_rows = df_tools_exploded[col_logs].apply(lambda x: x if isinstance(x, (dict, list)) else safe_json_parse(x)).tolist()
df_tools_flat = pd.json_normalize(tool_rows)

In [10]:
tool_whitelist = [
    'URL_fetch','body_fetch','return_fetch','tool','status','timestamp',
    'code_fetch','return_fetch.msg','return_fetch.message','return_fetch.data.client_rfc',
    'return_fetch.data.client_complete_name','return_fetch.data.program_status','return_fetch.data.program_name',
    'return_fetch.data.client_type','return_fetch.data.client_account','return_fetch.data.client_card',
    'return_fetch.data.client_name','return_fetch.httpCode','return_fetch.status','body_fetch.cas',
    'body_fetch.cancelMotive','return_fetch.success','return_fetch.results',' body_fetch.lat_base',
    'body_fetch.lng_base','return_fetch.response','body_fetch.firstDay','body_fetch.thirdShift',
    'body_fetch.idBeneficiary','body_fetch.thirdDay','body_fetch.secondIdService','body_fetch.firstShift',
    'body_fetch.firstIdService','body_fetch.secondDay','body_fetch.secondShift','body_fetch.account',
    'return_fetch.cas_folio','return_fetch.idAppointment'
]

In [11]:
cols_t = [c for c in tool_whitelist if c in df_tools_flat.columns]
df_tools_final = df_tools_flat[cols_t].copy()
df_tools_final.columns = [f"tool_{c.replace('.', '_')}" for c in df_tools_final.columns]

df_tools_final.index = df_tools_exploded.index
df_tools_merged = pd.concat([df_tools_exploded[['Id']], df_tools_final], axis=1)

print(f'{df_tools_merged.shape}')

(35, 38)


In [12]:
df_final = df_base.merge(df_tools_merged, on='Id', how='left')
df_final.columns = [c.replace(" ", "_").replace("(", "").replace(")", "").replace(".", "_") for c in df_final.columns]

In [13]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 71 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   Id                                           71 non-null     str    
 1   Id_Externo                                   71 non-null     str    
 2   Id_Canal                                     19 non-null     str    
 3   Canal                                        71 non-null     str    
 4   Timestamp                                    71 non-null     str    
 5   Inicio                                       71 non-null     str    
 6   Fin                                          71 non-null     str    
 7   Duración_s                                   71 non-null     int64  
 8   Análisis_Sentimental                         41 non-null     str    
 9   Tema_general_de_la_conversación              41 non-null     str    
 10  Resumen        

In [14]:
target_col = 'tool_return_fetch_response'
df_to_expand = df_final[['Id', 'tool_timestamp', target_col]].dropna(subset=[target_col]).copy()

df_res_exploded = df_to_expand.explode(target_col)

res_raw = df_res_exploded[target_col].apply(lambda x: x if isinstance(x, dict) else safe_json_parse(x)).tolist()
df_res_flat = pd.json_normalize(res_raw)

res_whitelist = ['estado', 'especialization', 'authority']
cols_presentes = [c for c in res_whitelist if c in df_res_flat.columns]
df_res_final = df_res_flat[cols_presentes].copy()
df_res_final.columns = [f"res_{c}" for c in df_res_final.columns]

df_sub_extraido = pd.concat([
    df_res_exploded[['Id', 'tool_timestamp']].reset_index(drop=True), 
    df_res_final.reset_index(drop=True)
], axis=1)

df_final_v2 = df_final.merge(df_sub_extraido, on=['Id', 'tool_timestamp'], how='left')

print(f"Sub-extracción exitosa. Columnas añadidas: {df_res_final.columns.tolist()}")
display(df_final_v2.head())

Sub-extracción exitosa. Columnas añadidas: ['res_estado', 'res_especialization', 'res_authority']


,Id,Id_Externo,Id_Canal,Canal,Timestamp,Inicio,Fin,Duración_s,Análisis_Sentimental,Tema_general_de_la_conversación,...,tool_body_fetch_firstShift,tool_body_fetch_firstIdService,tool_body_fetch_secondDay,tool_body_fetch_secondShift,tool_body_fetch_account,tool_return_fetch_cas_folio,tool_return_fetch_idAppointment,res_estado,res_especialization,res_authority
0,aa106f71-c43c-4483-8340-7d868d8d770a,,NaN,,1774286586000,2026-03-23T17:23:06.000Z,2026-03-23T17:25:44.959Z,158,neutral,cita dental,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,aa106f71-c43c-4483-8340-7d868d8d770a,,NaN,,1774286586000,2026-03-23T17:23:06.000Z,2026-03-23T17:25:44.959Z,158,neutral,cita dental,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,aa106f71-c43c-4483-8340-7d868d8d770a,,NaN,,1774286586000,2026-03-23T17:23:06.000Z,2026-03-23T17:25:44.959Z,158,neutral,cita dental,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,6a2c0317-58b3-4326-bf13-2e98dcdabfa9,,NaN,,1774286461000,2026-03-23T17:21:01.000Z,2026-03-23T17:21:42.770Z,41,neutral,Transferencia Asesor,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6a2c0317-58b3-4326-bf13-2e98dcdabfa9,,NaN,,1774286461000,2026-03-23T17:21:01.000Z,2026-03-23T17:21:42.770Z,41,neutral,Transferencia Asesor,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
columnas_sql_reales = [
    'Id', 'tool_timestamp', 'Id_Externo', 'Id_Canal', 'Canal', 'Timestamp', 
    'Inicio', 'Fin', 'Duración_s', 'Análisis_Sentimental', 
    'Tema_general_de_la_conversación', 'Resumen', 'Fue_resuelta', 
    'Fue_solo_agradecimiento', 'Es_saliente', 'Cerrada_por_inactividad', 
    'Fue_abandonada', 'Mensajes_del_asistente', 'Mensajes_del_usuario', 
    'Cantidad_de_preguntas_a_QNA', 'Herramientas_Usadas', 'Motivo_de_Cierre', 
    'Motivo_de_transferencia', 'ctx_creationRequestTimestamp', 
    'ctx_startedTime', 'ctx_startDelayMilliseconds', 'ctx_API_KEY', 
    'ctx_URL', 'ctx_tarjeta', 'ctx_siniestralidad', 'ctx_proxyData_recordFile', 
    'ctx_proxyData_roomId', 'ctx_proxyData_roomName', 'tool_URL_fetch', 
    'tool_body_fetch', 'tool_return_fetch', 'tool_tool', 'tool_status', 
    'tool_code_fetch', 'tool_return_fetch_msg', 'tool_return_fetch_message', 
    'tool_return_fetch_data_client_rfc', 'tool_return_fetch_data_client_complete_name', 
    'tool_return_fetch_data_program_status', 'tool_return_fetch_data_program_name', 
    'tool_return_fetch_data_client_type', 'tool_return_fetch_data_client_account', 
    'tool_return_fetch_data_client_card', 'tool_return_fetch_data_client_name', 
    'tool_return_fetch_httpCode', 'tool_return_fetch_status', 'tool_body_fetch_cas', 
    'tool_body_fetch_cancelMotive', 'tool_return_fetch_success', 
    'tool_body_fetch_lng_base', 'tool_body_fetch_firstDay', 
    'tool_body_fetch_thirdShift', 'tool_body_fetch_idBeneficiary', 
    'tool_body_fetch_thirdDay', 'tool_body_fetch_secondIdService', 
    'tool_body_fetch_firstShift', 'tool_body_fetch_firstIdService', 
    'tool_body_fetch_secondDay', 'tool_body_fetch_secondShift', 
    'tool_body_fetch_account', 'tool_return_fetch_cas_folio', 
    'tool_return_fetch_idAppointment', 'res_estado', 
    'res_especialization', 'res_authority'
]

# %%
# FILTRADO QUIRÚRGICO: Tiramos todo lo que no esté en la tabla de SQL
df_produccion = df_final_v2[[c for c in columnas_sql_reales if c in df_final_v2.columns]].copy()

print(f"✅ DataFrame alineado: {df_produccion.shape[1]} columnas listas para SQL.")

✅ DataFrame alineado: 70 columnas listas para SQL.


In [16]:
print(f"📊 Filas antes de limpiar: {len(df_produccion)}")

# Nos quedamos solo con la primera ocurrencia de cada par (Id, tool_timestamp)
df_produccion_clean = df_produccion.drop_duplicates(subset=['Id', 'tool_timestamp'], keep='first').copy()

print(f"✅ Filas después de limpiar: {len(df_produccion_clean)}")
if len(df_produccion) != len(df_produccion_clean):
    print(f"⚠️ Se eliminaron {len(df_produccion) - len(df_produccion_clean)} duplicados que iban a romper el SQL.")

📊 Filas antes de limpiar: 81
✅ Filas después de limpiar: 71
⚠️ Se eliminaron 10 duplicados que iban a romper el SQL.


In [21]:
df_produccion_clean.to_csv("resultado_final.csv", index=False)

In [22]:
def preparar_produccion_final(df):
    df_sql = df.copy()
    
    cols_num = ['Duración_s', 'Mensajes_del_asistente', 'Mensajes_del_usuario', 
                'ctx_startDelayMilliseconds', 'Cantidad_de_preguntas_a_QNA', 
                'Herramientas_Usadas', 'tool_code_fetch', 'tool_return_fetch_httpCode',
                'tool_body_fetch_lng_base', 'tool_body_fetch_secondIdService', 
                'tool_body_fetch_firstIdService', 'tool_return_fetch_idAppointment']
    
    cols_bit = ['Fue_resuelta', 'Fue_solo_agradecimiento', 'Es_saliente', 
                'Cerrada_por_inactividad', 'Fue_abandonada']

    for col in df_sql.columns:
        if col in cols_bit:
            df_sql[col] = pd.to_numeric(df_sql[col], errors='coerce').fillna(0).astype(int)
        elif col in cols_num:
            df_sql[col] = pd.to_numeric(df_sql[col], errors='coerce')
            df_sql[col] = df_sql[col].astype(object).where(pd.notnull(df_sql[col]), None)
        else:
            df_sql[col] = df_sql[col].astype(str).replace(['nan', 'None', 'NaN', 'null'], None)
            
    return df_sql

df_listo = preparar_produccion_final(df_produccion_clean)

In [23]:
df_para_sql = df_listo

conn_str = f"DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={os.getenv('DB_SERVER')},{os.getenv('DB_PORT')};DATABASE={os.getenv('BD')};UID={os.getenv('DB_USER')};PWD={os.getenv('DB_PASS')}"
conn = pyodbc.connect(conn_str)
cursor = conn.cursor()
cursor.fast_executemany = True 

TABLE_NAME = "dbo.inagent"

try:
    # A. Crear tabla de Staging con las columnas EXACTAS del DataFrame
    cursor.execute(f"IF OBJECT_ID('tempdb..#stg_inagent') IS NOT NULL DROP TABLE #stg_inagent")
    
    # Obtenemos las columnas actuales del DF
    cols = df_para_sql.columns.tolist()
    col_names_bracketed = ", ".join(f"[{c}]" for c in cols)
    
    # Creamos la temporal basada en la estructura de la real
    cursor.execute(f"SELECT TOP 0 {col_names_bracketed} INTO #stg_inagent FROM {TABLE_NAME}") 

    # B. Insert masivo usando el DataFrame PREPARADO
    placeholders = ", ".join("?" for _ in cols)
    sql_insert = f"INSERT INTO #stg_inagent ({col_names_bracketed}) VALUES ({placeholders})"
    
    # Convertimos a lista de tuplas desde el DF preparado (sin listas/dicts)
    data_to_load = [tuple(x) for x in df_para_sql.values]
    
    print(f"📦 Subiendo {len(data_to_load)} registros limpios a Staging...")
    cursor.executemany(sql_insert, data_to_load)
    
    # C. MERGE (Sincronización)
    sql_merge = f"""
    MERGE {TABLE_NAME} AS target
    USING #stg_inagent AS source
    ON (target.Id = source.Id AND target.tool_timestamp = source.tool_timestamp)
    WHEN MATCHED THEN
        UPDATE SET 
            target.Análisis_Sentimental = source.Análisis_Sentimental,
            target.Resumen = source.Resumen,
            target.tool_status = source.tool_status,
            target.res_estado = source.res_estado
    WHEN NOT MATCHED THEN
        INSERT ({col_names_bracketed})
        VALUES ({', '.join(f'source.[{c}]' for c in cols)});
    """
    
    print("🔄 Ejecutando MERGE en tabla definitiva...")
    cursor.execute(sql_merge)
    conn.commit()
    print(f"🚀 ÉXITO TOTAL: {len(df_para_sql)} registros sincronizados en {TABLE_NAME}.")

except Exception as e:
    conn.rollback()
    print(f"❌ Error en SQL: {e}")
finally:
    cursor.close()
    conn.close()

📦 Subiendo 71 registros limpios a Staging...
🔄 Ejecutando MERGE en tabla definitiva...
🚀 ÉXITO TOTAL: 71 registros sincronizados en dbo.inagent.
